# 04 Prompt Packing and Truncation

This notebook studies how to fit a multi-section prompt into a limited input budget.

Main formula:

`available_input_budget = context_window - reserved_output_tokens`

## Step 4 Goal

We want to pack important prompt sections first, then drop lower-priority sections if the input budget is too small.

Why this matters:
- RAG can add many chunks
- chat history grows over time
- tool outputs can be large
- agents often combine all of these at once

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parents[0]
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.append(str(PROJECT_ROOT / "src"))

from prompt_packer import (
    DEFAULT_CONTEXT_WINDOWS,
    DEFAULT_RESERVED_OUTPUT_TOKENS,
    PROMPT_PACKING_REPORT_PATH,
    PROMPT_SECTIONS_PATH,
    analyze_packing_across_tokenizers,
    build_packing_summary,
    create_packing_graphs,
    load_prompt_sections,
    save_packing_report,
)
from tokenizer_utils import FIGURES_DIR, REQUESTED_MODEL_NAMES, load_all_tokenizers

print("Project root:", PROJECT_ROOT)
print("Prompt sections path:", PROMPT_SECTIONS_PATH)
print("Packing report path:", PROMPT_PACKING_REPORT_PATH)
print("Figures directory:", FIGURES_DIR)

Project root: /Users/nikhiljuluri/Desktop/LLM_Infra_Projects/LLM_Inference_cost_Latency_analyser/week2_LLM_Inference
Prompt sections path: /Users/nikhiljuluri/Desktop/LLM_Infra_Projects/LLM_Inference_cost_Latency_analyser/week2_LLM_Inference/examples/prompt_sections.json
Packing report path: /Users/nikhiljuluri/Desktop/LLM_Infra_Projects/LLM_Inference_cost_Latency_analyser/week2_LLM_Inference/outputs/prompt_packing_report.csv
Figures directory: /Users/nikhiljuluri/Desktop/LLM_Infra_Projects/LLM_Inference_cost_Latency_analyser/week2_LLM_Inference/outputs/figures


## Load Prompt Sections

The prompt is broken into structured sections like system prompt, user question, chat history, retrieved chunks, and tool output.

In [2]:
sections = load_prompt_sections(PROMPT_SECTIONS_PATH)
sections

[{'name': 'system_prompt',
  'priority': 'required',
  'section_type': 'system',
  'order': 1,
  'text': 'You are an LLM infrastructure assistant. Be precise, preserve critical constraints, and explain tradeoffs clearly.'},
 {'name': 'developer_instruction',
  'priority': 'required',
  'section_type': 'instruction',
  'order': 2,
  'text': 'Always respect the requested output format. Prefer concise answers, mention uncertainty, and do not invent metrics that are not present in the prompt.'},
 {'name': 'user_question',
  'priority': 'required',
  'section_type': 'user',
  'order': 3,
  'text': 'Given retrieved documents, recent chat history, and tool output, summarize the likely inference bottlenecks and recommend the safest next step.'},
 {'name': 'chat_history_old',
  'priority': 'low',
  'section_type': 'history',
  'order': 4,
  'text': 'User: Last week we discussed GPU utilization, speculative decoding, batching, and cache hit rates in several earlier experiments. Assistant: We als

## Load Tokenizers

We reuse the same tokenizer-loading utility from previous steps.

In [3]:
tokenizers = load_all_tokenizers(REQUESTED_MODEL_NAMES)
list(tokenizers.keys())

['Qwen/Qwen2.5-1.5B-Instruct',
 'mistralai/Mistral-7B-Instruct-v0.2',
 'TinyLlama/TinyLlama-1.1B-Chat-v1.0']

## Run Packing Analysis

Packing order:
- required sections first
- then high priority
- then medium priority
- then low priority

If the budget runs out, lower-priority sections are dropped first.

In [4]:
results = analyze_packing_across_tokenizers(
    sections=sections,
    tokenizers=tokenizers,
    context_windows=DEFAULT_CONTEXT_WINDOWS,
    reserved_output_tokens=DEFAULT_RESERVED_OUTPUT_TOKENS,
)
df = save_packing_report(results, PROMPT_PACKING_REPORT_PATH)
df.head(10)

,tokenizer_name,context_window,reserved_output_tokens,available_input_budget,total_original_prompt_tokens,packed_prompt_tokens,remaining_input_tokens,included_sections,dropped_sections,number_of_included_sections,number_of_dropped_sections,fits_after_packing,required_sections_fit,packed_prompt_usage_percent,section_token_map
3,mistralai/Mistral-7B-Instruct-v0.2,4096,512,3584,3836,2621,963,"[system_prompt, developer_instruction, user_qu...",[rag_chunk_3],9,1,True,True,73.1306,"{'system_prompt': 21, 'developer_instruction':..."
6,TinyLlama/TinyLlama-1.1B-Chat-v1.0,4096,512,3584,3877,2663,921,"[system_prompt, developer_instruction, user_qu...",[rag_chunk_3],9,1,True,True,74.3025,"{'system_prompt': 22, 'developer_instruction':..."
0,Qwen/Qwen2.5-1.5B-Instruct,4096,512,3584,3371,3371,213,"[system_prompt, developer_instruction, user_qu...",[],10,0,True,True,94.0569,"{'system_prompt': 21, 'developer_instruction':..."
1,Qwen/Qwen2.5-1.5B-Instruct,8192,512,7680,3371,3371,4309,"[system_prompt, developer_instruction, user_qu...",[],10,0,True,True,43.8932,"{'system_prompt': 21, 'developer_instruction':..."
2,Qwen/Qwen2.5-1.5B-Instruct,32768,512,32256,3371,3371,28885,"[system_prompt, developer_instruction, user_qu...",[],10,0,True,True,10.4508,"{'system_prompt': 21, 'developer_instruction':..."
4,mistralai/Mistral-7B-Instruct-v0.2,8192,512,7680,3836,3836,3844,"[system_prompt, developer_instruction, user_qu...",[],10,0,True,True,49.9479,"{'system_prompt': 21, 'developer_instruction':..."
5,mistralai/Mistral-7B-Instruct-v0.2,32768,512,32256,3836,3836,28420,"[system_prompt, developer_instruction, user_qu...",[],10,0,True,True,11.8924,"{'system_prompt': 21, 'developer_instruction':..."
7,TinyLlama/TinyLlama-1.1B-Chat-v1.0,8192,512,7680,3877,3877,3803,"[system_prompt, developer_instruction, user_qu...",[],10,0,True,True,50.4818,"{'system_prompt': 22, 'developer_instruction':..."
8,TinyLlama/TinyLlama-1.1B-Chat-v1.0,32768,512,32256,3877,3877,28379,"[system_prompt, developer_instruction, user_qu...",[],10,0,True,True,12.0195,"{'system_prompt': 22, 'developer_instruction':..."


## Full Report Sorted by Dropped Sections

This shows which tokenizer and context window combinations lose the most prompt content.

In [5]:
df

,tokenizer_name,context_window,reserved_output_tokens,available_input_budget,total_original_prompt_tokens,packed_prompt_tokens,remaining_input_tokens,included_sections,dropped_sections,number_of_included_sections,number_of_dropped_sections,fits_after_packing,required_sections_fit,packed_prompt_usage_percent,section_token_map
3,mistralai/Mistral-7B-Instruct-v0.2,4096,512,3584,3836,2621,963,"[system_prompt, developer_instruction, user_qu...",[rag_chunk_3],9,1,True,True,73.1306,"{'system_prompt': 21, 'developer_instruction':..."
6,TinyLlama/TinyLlama-1.1B-Chat-v1.0,4096,512,3584,3877,2663,921,"[system_prompt, developer_instruction, user_qu...",[rag_chunk_3],9,1,True,True,74.3025,"{'system_prompt': 22, 'developer_instruction':..."
0,Qwen/Qwen2.5-1.5B-Instruct,4096,512,3584,3371,3371,213,"[system_prompt, developer_instruction, user_qu...",[],10,0,True,True,94.0569,"{'system_prompt': 21, 'developer_instruction':..."
1,Qwen/Qwen2.5-1.5B-Instruct,8192,512,7680,3371,3371,4309,"[system_prompt, developer_instruction, user_qu...",[],10,0,True,True,43.8932,"{'system_prompt': 21, 'developer_instruction':..."
2,Qwen/Qwen2.5-1.5B-Instruct,32768,512,32256,3371,3371,28885,"[system_prompt, developer_instruction, user_qu...",[],10,0,True,True,10.4508,"{'system_prompt': 21, 'developer_instruction':..."
4,mistralai/Mistral-7B-Instruct-v0.2,8192,512,7680,3836,3836,3844,"[system_prompt, developer_instruction, user_qu...",[],10,0,True,True,49.9479,"{'system_prompt': 21, 'developer_instruction':..."
5,mistralai/Mistral-7B-Instruct-v0.2,32768,512,32256,3836,3836,28420,"[system_prompt, developer_instruction, user_qu...",[],10,0,True,True,11.8924,"{'system_prompt': 21, 'developer_instruction':..."
7,TinyLlama/TinyLlama-1.1B-Chat-v1.0,8192,512,7680,3877,3877,3803,"[system_prompt, developer_instruction, user_qu...",[],10,0,True,True,50.4818,"{'system_prompt': 22, 'developer_instruction':..."
8,TinyLlama/TinyLlama-1.1B-Chat-v1.0,32768,512,32256,3877,3877,28379,"[system_prompt, developer_instruction, user_qu...",[],10,0,True,True,12.0195,"{'system_prompt': 22, 'developer_instruction':..."


## Summary Findings

These values answer the main Step 4 questions:
- which sections are dropped most often
- which context windows keep all sections
- which tokenizer uses more tokens for the same prompt
- whether required sections still fit
- how much input budget remains after packing

In [6]:
summary = build_packing_summary(df)
summary

{'most_often_dropped_section': 'rag_chunk_3',
 'context_windows_that_keep_all_sections': [8192, 32768],
 'highest_tokenizer_for_same_prompt': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0',
 'lowest_tokenizer_for_same_prompt': 'Qwen/Qwen2.5-1.5B-Instruct',
 'required_sections_fit_by_context_window': {4096: True,
  8192: True,
  32768: True},
 'average_remaining_input_tokens_by_context_window': {4096: 699.0,
  8192: 3985.33,
  32768: 28561.33}}

## Create Graphs

The graphs help us see how much content survives packing and how token usage changes by section and context window.

In [7]:
create_packing_graphs(df, FIGURES_DIR)
sorted(path.name for path in FIGURES_DIR.glob('*.png'))

['character_count_vs_token_count.png',
 'context_usage_by_prompt.png',
 'context_used_percent_by_tokenizer.png',
 'fits_vs_exceeds_context.png',
 'included_vs_dropped_sections.png',
 'packed_prompt_usage.png',
 'remaining_tokens_by_prompt.png',
 'strategy_comparison.png',
 'token_budget_by_section.png',
 'token_count_by_category_and_tokenizer.png',
 'tokenizer_comparison_summary.png',
 'tokens_per_word_by_category_and_tokenizer.png']

## Why Prompt Packing Matters

Prompt packing is important because long prompts are made of many competing parts.

- **RAG** adds retrieved chunks quickly.
- **Chat history** grows turn by turn.
- **Agents** may include system rules, tool schemas, tool results, and memory.
- **Tool calling** often introduces large JSON sections.
- **Long-context applications** must decide what to keep when everything cannot fit.

Some sections should almost never be dropped:
- system prompt
- user question
- final instruction

Those sections define behavior, task, and output requirements.

Old chat history is usually lower priority than recent history because the newest turns are more likely to affect the current answer.

## What the Graphs Show

1. **Included vs dropped sections**
   This shows how many sections survive packing for each context window.

2. **Token usage by section**
   This highlights which sections consume the most input budget.

3. **Packed prompt usage percent**
   This compares how much of the available input budget is used after packing.

4. **Strategy comparison**
   This shows how dropping behavior changes as the context window gets larger.

## Step 4 Takeaway

Prompt packing is really a budgeting problem. The best strategy is not to keep everything equally, but to keep the most important sections first and drop lower-value sections when the budget gets tight.